In [2]:

import torch
import bisect
import numpy as np
from numpy.dtypes import StringDType
import csv
from random import randrange
import subprocess
import mmap

In [3]:
def countLines (file_path) -> int:
  line_count = 0
  with open(file_path, "r") as python_filehandle:
    with mmap.mmap(python_filehandle.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_filehandle:
        while mmap_filehandle.readline():
            line_count += 1

  return line_count

In [4]:
def retrieveSentence (sentence_idx, tokens_list, sentence_offsets) -> str:
    end_idx = len(tokens_list) if sentence_idx+1 == len(sentence_offsets) else sentence_offsets[sentence_idx+1]
    print(end_idx)
    return "".join(tokens_dict[tokno].replace("<wb>", " ") for tokno in tokens_list[sentence_offsets[sentence_idx]:end_idx]).strip()

In [5]:
def retrieveSubtext(subtext_idx, tokens_list, subtext_offsets) -> str:
    end_idx = len(tokens_list) if subtext_idx+1 == len(subtext_offsets) else subtext_offsets[subtext_idx+1]
    return "".join(tokens_dict[tokno].replace("<wb>", " ") for tokno in tokens_list[subtext_offsets[subtext_idx]:end_idx]).strip()

In [6]:
def retrieveSubtextBeginning(subtext_idx, tokens_list, subtext_offsets) -> str:
    end_idx = subtext_offsets[subtext_idx] + 15
    return "".join(tokens_dict[tokno].replace("<wb>", " ") for tokno in tokens_list[subtext_offsets[subtext_idx]:end_idx]).strip()

In [7]:
def stringifyTokensTensor(tokens_tensor, token_boundaries=False) -> str:
    token_separator = "" if token_boundaries == False else "|"
    return token_separator.join(tokens_dict[tokno].replace("<wb>", " ") for tokno in tokens_tensor.tolist()).strip()

In [8]:
def sortedListFind(sorted_list, sought_after_value) -> int:
    'Locate the leftmost value exactly equal to x'
    i = bisect.bisect_left(sorted_list, sought_after_value)
    if i != len(sorted_list) and sorted_list[i] == sought_after_value:
        return i
    else:
        return -1

In [9]:
def getWindowLossMask(subtext_idx, window_idx) -> tuple(torch.Tensor, int, int):
    token_offset_at_window_start = subtext_offsets[subtext_idx] + window_idx*32

    first_whole_word_idx = 0
    start_word_offset = 0
    for i in range(0, 32):
        start_word_offset = sortedListFind(word_offsets, token_offset_at_window_start+i)
        if start_word_offset != -1:
            first_whole_word_idx = i
            break
    last_whole_word_idx = 31
    for i in range(31, 64):
        idx_pos_in_word_offsets = sortedListFind(word_offsets, token_offset_at_window_start+i+1)
        if idx_pos_in_word_offsets != -1 or idx_pos_in_word_offsets == len(word_offsets):
            last_whole_word_idx = i
            break
    
    loss_mask = torch.zeros(64, dtype=torch.bool)
    loss_mask = subtext_windows[subtext_idx][window_idx] != 0 #sets padding token positions to False
    loss_mask[last_whole_word_idx + 1:] = False
    loss_mask[:first_whole_word_idx] = False

    #to account for cases where the window contains only non-initial subword tokens and padding
    if loss_mask.nonzero().size(0) == 0:
        return loss_mask, -1, -1
    
    post_final_word_token_offset = token_offset_at_window_start + loss_mask.nonzero()[-1].item() + 1
    end_word_offset = 0
    post_final_word_word_offset = sortedListFind(word_offsets, post_final_word_token_offset)
    if post_final_word_word_offset == -1:
        end_word_offset = len(word_offsets) - 1
    else:
        end_word_offset = post_final_word_word_offset - 1

    return loss_mask, start_word_offset, end_word_offset

In [10]:
def getWindowWordLengths(start_word_offset, end_word_offset, padded=False, max_word_length=32):
    subtext_window_word_lengths = []
    for i in range(start_word_offset, end_word_offset):
        word_token_length = word_offsets[i + 1] - word_offsets[i]
        subtext_window_word_lengths.append(word_token_length)
    
    final_word_token_length = 1
    if len(word_offsets) - 1 == end_word_offset:
        final_word_token_length = len(tokens_list) - word_offsets[end_word_offset]
    else:
        final_word_token_length = word_offsets[end_word_offset + 1] - word_offsets[end_word_offset]
    subtext_window_word_lengths.append(final_word_token_length)

    if padded:
        return torch.nn.functional.pad(torch.tensor(subtext_window_word_lengths, dtype=torch.int64), (0, max_word_length-len(subtext_window_word_lengths)), value=0) 
    else:
        return torch.tensor(subtext_window_word_lengths, dtype=torch.int64)

In [11]:
batch_size = 32

In [12]:
def getCharLSTMInputWindow(token_window: torch.Tensor, loss_mask: torch.Tensor, start_word_offset: int, end_word_offset: int, max_word_length=32) -> (torch.Tensor, torch.Tensor, list):

    if start_word_offset == -1:
        return torch.zeros(32, 32, dtype=torch.int64), torch.zeros(32, dtype=torch.bool), []
    
    word_tokens_tensor = token_window[loss_mask]
    
    i = start_word_offset
    j = start_word_offset
    char_lstm_window_word_tensors_list = []
    word_lengths_list = []
    for i in range(start_word_offset, end_word_offset):
        word_token_length = word_offsets[i + 1] - word_offsets[i]
        start_idx = j - start_word_offset
        one_past_end_idx = start_idx + word_token_length

        word_str = stringifyTokensTensor(word_tokens_tensor[start_idx:one_past_end_idx])[:max_word_length]
        word_char_ids = []
        for char in word_str:
            if char in char_dict_reversed:
                word_char_ids.append(char_dict_reversed[char])
            else:
                word_char_ids.append(char_dict_reversed["<unk>"])

        word_tensor = torch.nn.functional.pad(torch.tensor(word_char_ids, dtype=torch.int64), (0, max_word_length-len(word_char_ids)), value=0)
        char_lstm_window_word_tensors_list.append(word_tensor)
        word_lengths_list.append(len(word_char_ids))
        j += word_token_length

    final_word_token_length = 1
    if len(word_offsets) -1 == end_word_offset:
        final_word_token_length = len(tokens_list) - word_offsets[end_word_offset]
    else:
        final_word_token_length = word_offsets[end_word_offset + 1] - word_offsets[end_word_offset]
    final_start_idx = j - start_word_offset
    final_one_past_end_idx = final_start_idx + final_word_token_length
    word_str = stringifyTokensTensor(word_tokens_tensor[final_start_idx:final_one_past_end_idx])[:max_word_length]
    word_char_ids = []
    for char in word_str:
        if char in char_dict_reversed:
            word_char_ids.append(char_dict_reversed[char])
        else:
            word_char_ids.append(char_dict_reversed["<unk>"])

    
    word_tensor = torch.nn.functional.pad(torch.tensor(word_char_ids, dtype=torch.int64), (0, max_word_length-len(word_char_ids)), value=0)
    char_lstm_window_word_tensors_list.append(word_tensor)
    word_lengths_list.append(len(word_char_ids))
    
    unpadded_char_lstm_window_tensor = torch.stack(char_lstm_window_word_tensors_list, dim=0)
    char_lstm_window_tensor = torch.nn.functional.pad(unpadded_char_lstm_window_tensor, (0, 0, 0, 32-unpadded_char_lstm_window_tensor.size(0)), value=0)
    lstm_word_mask = (char_lstm_window_tensor != 0).any(dim=1)
    
    return char_lstm_window_tensor, lstm_word_mask, word_lengths_list

In [13]:
token_vocab_length = countLines("bpe_token_indices.csv")

In [14]:
bpe_token_indices_file = open("bpe_token_indices.csv", "r")
tokenised_chu_words_training_file = open("tokenised_chu_words_training_deepcleaned.csv", "r")

In [15]:
tokens_list = []
word_offsets = []
with mmap.mmap(tokenised_chu_words_training_file.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_tokens_file:
  word_token_count = 0
  for line in iter(mmap_tokens_file.readline, b""):
    word_offsets.append(word_token_count)
    for token_no in line.decode("utf-8").strip().split(",")[0].split(" "):
      tokens_list.append(int(token_no))
      word_token_count += 1

In [16]:
sentence_offsets = []
subtext_offsets = []
text_offsets = []
row_no = 0
sentence_no_prev = 0
subtext_no_prev = 0
text_id_prev = 0
token_count = 0
for row in csv.DictReader(open("../../chu_words_tagged.csv", "r"), delimiter="|"):
    sentence_no = int(row["sentence_no"])
    subtext_no = int(row["subtitle_id"])
    text_id_no = int(row["text_id"])
    if sentence_no != sentence_no_prev:
        sentence_offsets.append(word_offsets[row_no])
        sentence_no_prev = sentence_no
    if text_id_no != text_id_prev:
        text_offsets.append(word_offsets[row_no])
        text_id_prev = text_id_no
        subtext_offsets.append(word_offsets[row_no])
        subtext_no_prev = subtext_no
    elif subtext_no != subtext_no_prev:
        subtext_offsets.append(word_offsets[row_no])
        subtext_no_prev = subtext_no
    row_no += 1

In [17]:
len(tokens_list), len(word_offsets), len(sentence_offsets), len(subtext_offsets), len(text_offsets)

(347761, 242536, 27241, 407, 9)

In [18]:
tokens_dict = {}
tokens_dict_reversed = {}
char_dict = {}
char_dict_reversed = {}
with mmap.mmap(bpe_token_indices_file.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_bpe_file:
    line_no = 0
    for bin_line in iter(mmap_bpe_file.readline, b""):
        line = bin_line.decode("utf-8").strip()
        split_line = line.split(",")
        tokens_dict[int(split_line[0])] = split_line[1].strip()
        tokens_dict_reversed[split_line[1]] = int(split_line[0])
        if line_no < 38:
            char_dict[int(split_line[0])] = split_line[1].strip()
            char_dict_reversed[split_line[1]] = int(split_line[0])
        line_no += 1

In [19]:
sentence_offsets_tensor = torch.tensor(sentence_offsets, dtype=torch.int64)
tensor_snt_lngths = torch.diff(sentence_offsets_tensor)
print("Max sentence length:", tensor_snt_lngths.max())
print("Median sentence length:", tensor_snt_lngths.median())

Max sentence length: tensor(291)
Median sentence length: tensor(10)


In [20]:
tokens_tensor = torch.tensor(tokens_list, dtype=torch.int64)

In [21]:
subtext_windows = []
flat_subtext_window_tensors = []
for i in range(len(subtext_offsets)):
    end_idx = len(tokens_list) if i+1 == len(subtext_offsets) else subtext_offsets[i+1]
    subtext_tokens = tokens_tensor[subtext_offsets[i]:end_idx]
    subtext_token_length = subtext_tokens.size(0)
    # leftover = subtext_token_length
    # window_length = 0
    subtext_chunks = []
    for j in range(0, subtext_token_length, 32):
        window_tokens = subtext_tokens[j:j+64]
        subtext_chunks.append(torch.nn.functional.pad(window_tokens, (0, 64-window_tokens.size(0)), value=0))
        flat_subtext_window_tensors.append(torch.nn.functional.pad(window_tokens, (0, 64-window_tokens.size(0)), value=0))
    subtext_windows.append(torch.stack(subtext_chunks, dim=0))

flat_subtext_window_tensors = torch.stack(flat_subtext_window_tensors)

subtext_window_sizes = []
for subtext_tensors in subtext_windows:
    subtext_window_sizes.append(subtext_tensors.size(0))

In [22]:
subtext_window_loss_masks = []
flat_token_loss_mask_tensors = []
flat_window_word_token_lengths = []
flat_window_word_token_lengths_padded_tensor = []
for s in range(len(subtext_windows)):
    loss_mask_chunks = []
    for w in range(subtext_windows[s].size(0)):
        loss_mask, start_word_offset, end_word_offset = getWindowLossMask(s, w)
        loss_mask_chunks.append((loss_mask, start_word_offset, end_word_offset))
        flat_token_loss_mask_tensors.append(loss_mask)
        flat_window_word_token_lengths.append(getWindowWordLengths(start_word_offset, end_word_offset))
        flat_window_word_token_lengths_padded_tensor.append(getWindowWordLengths(start_word_offset, end_word_offset, padded=True))
    subtext_window_loss_masks.append(loss_mask_chunks)
flat_token_loss_mask_tensors = torch.stack(flat_token_loss_mask_tensors)
flat_window_word_token_lengths_padded_tensor = torch.stack(flat_window_word_token_lengths_padded_tensor)


In [23]:
print(stringifyTokensTensor(flat_subtext_window_tensors[5000], True))
flat_subtext_window_tensors[5000], flat_window_word_token_lengths[5000], flat_window_word_token_lengths_padded_tensor[5000]

ѧ| вь| темніцѫ| ідѣаше| же| вьслѣ|дъ| народъ| многъ| ѣкоже| пль|номъ| бъіті| вьсѣмъ| стъ|гна|м| глаголаахѫ| ц|іі| о| піоніі| како| прісно| б|лѣ|дъ| съі| нъінѣ| роу|мѣ|но| ліце| емоу| естъ| дръжѧ|шті| же| его| саві|ні| за| різъі| отъ|ръва|ніѣ| раді| отъ| народа| глаголаахѫ| ц|іі| на|рѫ|га|ѭште| сѧ| те|тъ|ка| бо|ітъ| сѧ| да


(tensor([ 225,   81, 3399, 1619,   71, 3937,  113,  956, 2087,  308,  972,  687,
          503,  877, 2015, 3356,   14, 3953,  186,  841,   47, 2813,  376, 1199,
           49,  104,  113,  576,  457, 2845,  181,   85,  822,  144,  161, 3434,
          166,   71,  135, 2097,   60,  171, 1290,   84, 3672,  297,  368,   84,
         1674, 3953,  186,  841,   68,  316,  212,  393,   70,  238,   43,  134,
          112, 2315,   70,  105]),
 tensor([1, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 3, 1, 2, 1, 1, 1, 1, 3, 1, 1, 3]),
 tensor([1, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 3, 1, 2, 1, 1, 1, 1, 3, 1, 1, 3, 0,
         0, 0, 0, 0, 0, 0, 0, 0]))

In [24]:
lstm_windows = []
lstm_word_masks = []
lstm_word_lengths = []
for s in range(len(subtext_windows)):
    subtext_char_lstm_window_tuples = []
    for w in range(subtext_windows[s].size(0)):
        loss_mask, start_word_offset, end_word_offset = subtext_window_loss_masks[s][w]
        lstm_window, lstm_word_mask, lstm_window_word_lengths = getCharLSTMInputWindow(subtext_windows[s][w], loss_mask, start_word_offset, end_word_offset)
        lstm_windows.append(lstm_window)
        lstm_word_masks.append(lstm_word_mask)
        lstm_word_lengths.append(torch.tensor(lstm_window_word_lengths, dtype=torch.int64))

lstm_windows = torch.stack(lstm_windows)
lstm_word_masks = torch.stack(lstm_word_masks)

In [25]:
class textWindowsDataset(torch.utils.data.Dataset):
    def __init__(self, token_windows, token_window_loss_masks, token_window_word_lengths, lstm_windows, lstm_word_masks, lstm_word_lengths):
        self.token_windows = token_windows
        self.token_window_loss_masks = token_window_loss_masks
        self.token_window_word_lengths = token_window_word_lengths
        self.lstm_windows = lstm_windows
        self.lstm_word_masks = lstm_word_masks
        self.lstm_word_lengths = lstm_word_lengths

    def __len__(self):
        return len(self.token_windows)

    def __getitem__(self, idx):
        
        return {'token_windows': self.token_windows[idx], 'token_window_loss_masks': self.token_window_loss_masks[idx], 'token_window_word_lengths': self.token_window_word_lengths[idx], 'lstm_windows': self.lstm_windows[idx], 'lstm_word_masks': self.lstm_word_masks[idx], 'lstm_word_lengths': self.lstm_word_lengths[idx]}

In [26]:
mydataset = textWindowsDataset(flat_subtext_window_tensors, flat_token_loss_mask_tensors, flat_window_word_token_lengths_padded_tensor, lstm_windows, lstm_word_masks, lstm_word_lengths)
mydataset[3000]['lstm_windows'][:, :10], mydataset[3000]['lstm_word_masks'], stringifyTokensTensor(mydataset[3000]['token_windows'])

(tensor([[23, 10, 33, 28, 17,  0,  0,  0,  0,  0],
         [12, 35, 31, 33, 36,  3,  0,  0,  0,  0],
         [23, 24,  0,  0,  0,  0,  0,  0,  0,  0],
         [23, 36, 12, 35,  9, 12, 35,  0,  0,  0],
         [14, 12, 11, 14, 12, 35,  0,  0,  0,  0],
         [34, 17, 23, 16,  0,  0,  0,  0,  0,  0],
         [21, 17, 34, 12, 23, 32,  6,  0,  0,  0],
         [16,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [18, 11, 23, 11, 36, 25, 11,  0,  0,  0],
         [18,  6, 19, 34, 21, 17, 34, 12, 35,  3],
         [23, 24,  0,  0,  0,  0,  0,  0,  0,  0],
         [13, 12, 23, 32, 16,  0,  0,  0,  0,  0],
         [23,  6, 14, 33, 21, 11, 10,  6, 16, 24],
         [12, 32,  6, 18, 21, 17, 32, 16,  0,  0],
         [36, 16, 26, 11,  0,  0,  0,  0,  0,  0],
         [32, 18, 12, 11,  0,  0,  0,  0,  0,  0],
         [12, 32,  6,  0,  0,  0,  0,  0,  0,  0],
         [28, 21, 33,  9,  6,  0,  0,  0,  0,  0],
         [14, 12, 16,  9,  6,  0,  0,  0,  0,  0],
         [16,  0,  0,  0,  0,  

In [27]:
subtext_windows[20][17], stringifyTokensTensor(subtext_windows[20][17]), subtext_window_loss_masks[20][17]

(tensor([  62,   39, 1668,  174,  178,  311, 1618, 2679,  222,  237, 3730,   60,
           99,   39, 3287, 1870,  582,  344, 1144, 2018,  161,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0]),
 'не і мъітаре лі тако творѧтъ бѫдѣте оубо въі съвръшені ѣко і отецъ вашъ нбскъі съвръшенъ естъ<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>',
 (tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
           True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
           True, False, False, False, False, False, False, False, False, False,
       

In [28]:
class MorphologyLSTMTransformerModel(torch.nn.Module):
    def __init__(self, token_vocab_size=4539, token_embedding_dim=256, token_seq_length=64, attention_heads=4, trans_layers=4, char_vocab_size=38, char_embedding_dim=32, lstm_hidden_size=64, lstm_layers=1):
        super().__init__()

        
        self.token_embedder = torch.nn.Embedding(num_embeddings=token_vocab_size, embedding_dim=token_embedding_dim, padding_idx=0)
        self.positional_embedder = torch.nn.Embedding(num_embeddings=token_seq_length, embedding_dim=token_embedding_dim)
        transformer_encoder_layer = torch.nn.TransformerEncoderLayer(d_model=token_embedding_dim, nhead=attention_heads, dim_feedforward=4*token_embedding_dim, batch_first=True)
        self.transformer = torch.nn.TransformerEncoder(transformer_encoder_layer, trans_layers)

        self.char_embedder = torch.nn.Embedding(num_embeddings=char_vocab_size, embedding_dim=char_embedding_dim, padding_idx=0)
        self.char_dropout = torch.nn.Dropout(0.1)
        self.lstm = torch.nn.LSTM(input_size=char_embedding_dim, hidden_size=lstm_hidden_size, num_layers=lstm_layers, batch_first=True, bidirectional=True)

        self.lstm_hidden_size = lstm_hidden_size
        self.register_buffer("position_ids", torch.arange(token_seq_length).unsqueeze_(0))
        self.register_buffer("word_offsets", torch.tensor(word_offsets))

    def poolWordTokens(self, token_windows, token_windows_loss_masks, token_windows_word_lengths):

        B, W, H = token_windows.shape
        L = W//2 #L is the char lstm's max words per window, which is always half of the transformer's token-window size
        
        flattened_selected_word_tokens = token_windows[token_windows_loss_masks]
        flattened_unpadded_lengths = token_windows_word_lengths[token_windows_word_lengths > 0]
    
        flattened_pooled_tokens = torch.segment_reduce(flattened_selected_word_tokens, 'mean', lengths=flattened_unpadded_lengths)
    
        word_counts_per_window = (token_windows_word_lengths > 0).sum(dim=1)
    
        rebuilt_pooled_windows = token_windows.new_zeros(B, L, H) #new_zeros() just inherits properties like device from the tensor you call it on; it still creates a new tensor and doesn't modify token_windows
    
        row_idx = torch.arange(B, device=token_windows.device).unsqueeze(1).expand(-1, L)
        col_idx = torch.arange(L, device=token_windows.device).unsqueeze(0).expand(B, -1)
    
        keep = col_idx < word_counts_per_window.unsqueeze(1) #returns a boolean mask of shape (B, L) that for each row has as many Trues as are in the corresponding entry of word_counts_per_window, with the rest being False
    
        rebuilt_pooled_windows[row_idx[keep], col_idx[keep]] = flattened_pooled_tokens
    
        return rebuilt_pooled_windows
    
    def forward(self, token_windows, token_windows_loss_masks, token_windows_word_lengths, lstm_windows, lstm_word_masks, lstm_word_lengths) -> torch.Tensor:
        ### TRANSFORMER ###
        token_embeddings = self.token_embedder(token_windows)
        positional_embeddings = self.positional_embedder(self.position_ids.expand(token_embeddings.size(0), -1))
        padding_mask = (token_windows == 0)
        
        token_states = self.transformer(token_embeddings + positional_embeddings, src_key_padding_mask=padding_mask)
        word_pooled_token_states = self.poolWordTokens(token_states, token_windows_loss_masks, token_windows_word_lengths)

        ### LSTM ###
        B, W, L = lstm_windows.shape
        flat_lstm_windows = lstm_windows.reshape(B*W, L)
        flat_lstm_word_masks = lstm_word_masks.reshape(B*W)
        flat_unwordpadded_lstm_windows = flat_lstm_windows[flat_lstm_word_masks]
        
        lstm_char_embeddings = self.char_dropout(self.char_embedder(flat_unwordpadded_lstm_windows))
        packed_char_embeddings = torch.nn.utils.rnn.pack_padded_sequence(lstm_char_embeddings, torch.cat(lstm_word_lengths), batch_first=True, enforce_sorted=False)

        packed_output, (h_n, c_n) = self.lstm(packed_char_embeddings)
        flat_unpadded_word_vectors = torch.cat([h_n[0], h_n[1]], dim=1)
        
        flat_padded_word_vectors = flat_unpadded_word_vectors.new_zeros(B*W, self.lstm_hidden_size*2)
        flat_padded_word_vectors[flat_lstm_word_masks] = flat_unpadded_word_vectors
        
        final_lstm_word_vectors = flat_padded_word_vectors.reshape(B, W, self.lstm_hidden_size*2)
                                                                         
        combined_final_word_vectors = torch.cat([word_pooled_token_states, final_lstm_word_vectors], dim=-1)                                                                  
        
        return combined_final_word_vectors

In [29]:
char_embedder = torch.nn.Embedding(num_embeddings=38, embedding_dim=32, padding_idx=0)
char_dropout = torch.nn.Dropout(0.1)
lstm = torch.nn.LSTM(input_size=32, hidden_size=64, num_layers=1, batch_first=True, bidirectional=True)
dt = mydataset[:32]
lstm_wndws = dt['lstm_windows']
lstm_msks = dt['lstm_word_masks']
lstm_lngths = dt['lstm_word_lengths']

In [30]:
B, W, L = lstm_wndws.shape
flat_lstm_wndws = lstm_wndws.reshape(B*W, L)
flat_lstm_msks = lstm_msks.reshape(B*W)

flat_unwordpadded_lstm_wndws = flat_lstm_wndws[flat_lstm_msks]

lstm_embds = char_dropout(char_embedder(flat_unwordpadded_lstm_wndws))
pcked_lstm_embds = torch.nn.utils.rnn.pack_padded_sequence(lstm_embds, torch.cat(lstm_lngths), batch_first=True, enforce_sorted=False)

packed_output, (h_n, c_n) = lstm(pcked_lstm_embds)

In [31]:
flat_unpadded_word_vectors = torch.cat([h_n[0], h_n[1]], dim=1)
flat_padded_word_vectors = flat_unpadded_word_vectors.new_zeros(B*W, 128)
flat_padded_word_vectors[flat_lstm_msks] = flat_unpadded_word_vectors

final_word_vectors = flat_padded_word_vectors.reshape(B, W, 128)

stringifyTokensTensor(mydataset[1]['token_windows'][mydataset[1]['token_window_loss_masks']]), lstm_wndws[1][:, :12], final_word_vectors[1][:, :5]


('погані сѫште кръстівше же сѧ рімскамі і гръчьскъімі пісменъі нѫждаахѫ сѧ словѣнскъі рѣчь безь оустроеніа нѫ како',
 tensor([[37, 12, 28, 17, 10, 16,  0,  0,  0,  0,  0,  0],
         [23, 30,  2, 32, 11,  0,  0,  0,  0,  0,  0,  0],
         [13, 21,  6, 23, 32, 16, 18,  2, 11,  0,  0,  0],
         [15, 11,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [23, 24,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [21, 16, 14, 23, 13, 17, 14, 16,  0,  0,  0,  0],
         [16,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [28, 21,  6,  8, 25, 23, 13,  6, 16, 14, 16,  0],
         [37, 16, 23, 14, 11, 10,  6, 16,  0,  0,  0,  0],
         [10, 30, 15, 34, 17, 17,  9, 30,  0,  0,  0,  0],
         [23, 24,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [23, 36, 12, 18, 33, 10, 23, 13,  6, 16,  0,  0],
         [21, 33,  8, 25,  0,  0,  0,  0,  0,  0,  0,  0],
         [31, 11, 19, 25,  0,  0,  0,  0,  0,  0,  0,  0],
         [12, 35, 23, 32, 21, 12, 11, 10, 16, 17,  0,  0]

In [32]:
def poolWordTokensBatch(token_windows, token_windows_loss_masks, token_windows_word_lengths):

    #B, W, H = token_windows.shape
    B, W = token_windows.shape
    
    flattened_selected_word_tokens = token_windows[token_windows_loss_masks]
    flattened_unpadded_lengths = token_windows_word_lengths[token_windows_word_lengths > 0]

    flattened_pooled_tokens = torch.segment_reduce(flattened_selected_word_tokens, 'mean', lengths=flattened_unpadded_lengths)

    word_counts_per_window = (token_windows_word_lengths > 0).sum(dim=1)

    #rebuilt_pooled_windows = token_windows.new_zeros(B, W, H) #new_zeros() just inherits properties like device from the tensor you call it on; it still creates a new tensor and doesn't modify token_windows
    rebuilt_pooled_windows = token_windows.new_zeros(B, W) #new_zeros() just inherits properties like device from the tensor you call it on; it still creates a new tensor and doesn't modify token_windows

    row_idx = torch.arange(B, device=token_windows.device).unsqueeze(1).expand(-1, W)
    col_idx = torch.arange(W, device=token_windows.device).unsqueeze(0).expand(B, -1)

    keep = col_idx < word_counts_per_window.unsqueeze(1) #returns a boolean mask of shape (B, W) that for each row has as many Trues as are in the corresponding entry of word_counts_per_window, with the rest being False

    rebuilt_pooled_windows[row_idx[keep], col_idx[keep]] = flattened_pooled_tokens

    return rebuilt_pooled_windows

    
    

In [36]:
dta = mydataset[0:32]
tkn_wndw = dta['token_windows']
tkn_msks = dta['token_window_loss_masks']
tkn_lngths = dta['token_window_word_lengths']
lstm_wndws = dta['lstm_windows']
lstm_msks = dta['lstm_word_masks']
lstm_lngths = dta['lstm_word_lengths']

network = MorphologyLSTMTransformerModel()
#output = network(tkn_wndw, tkn_msks, tkn_lngths, lstm_wndws, lstm_msks, lstm_lngths)
output = network(tkn_wndw, tkn_msks, tkn_lngths, lstm_wndws, lstm_msks, lstm_lngths)




In [39]:
output[4, :, :5], stringifyTokensTensor(tkn_wndw[4][tkn_msks[4]])

(tensor([[ 1.3356,  0.4748, -1.5656, -0.1456, -0.4395],
         [-0.3775,  0.2738, -2.5933,  0.4536, -1.7356],
         [ 0.0194,  0.0571, -1.5343, -0.7886,  0.3103],
         [ 1.0445,  0.5908, -0.4249, -0.4221,  0.1445],
         [ 0.9439,  0.2240, -0.8155, -0.2063,  0.5678],
         [ 1.2802,  2.1308, -0.2231,  1.1498,  0.1175],
         [ 1.0887,  0.6081, -1.4851, -0.1049, -0.6552],
         [ 0.4668, -0.0198, -0.9567,  0.1074, -0.2202],
         [ 2.2030, -0.3997, -2.9073, -0.7326,  0.1583],
         [-0.3213,  1.7723,  0.2248, -0.4795, -0.0493],
         [ 1.2188, -0.6324, -2.5873, -1.9383, -1.7608],
         [ 0.7094,  0.2094, -1.0896, -1.4424,  0.3962],
         [ 0.4434,  1.0875, -2.7201, -0.6985, -1.0496],
         [ 0.4329,  0.4165, -3.4324,  0.8309, -1.1453],
         [ 1.0589,  0.9645, -0.4654,  0.0504, -0.7262],
         [ 0.3004,  0.6879, -1.3857, -0.8418, -0.7942],
         [ 0.0531,  0.6487, -0.6238, -0.4948, -0.1713],
         [ 1.0659,  1.1164, -1.1088, -0.1983, -0

In [35]:
def poolWorkTokenVectors(token_window: torch.Tensor, loss_mask: torch.Tensor, start_word_offset: int, end_word_offset: int) -> torch.Tensor:

    word_tokens_tensor = token_window[loss_mask]
    #word_tokens_tensor = token_window[loss_mask].float()

    pooled_tensors_list = []
    #deliberately leave off the last word so we can deal with the possibility of it being the last ever word in the set
    i = start_word_offset
    j = start_word_offset
    for i in range(start_word_offset, end_word_offset):
        word_token_length = word_offsets[i + 1] - word_offsets[i]
        start_idx = j - start_word_offset
        one_past_end_idx = start_idx + word_token_length
        pooled_tensor = word_tokens_tensor[start_idx:one_past_end_idx].mean(dim=0)
        #print(start_idx, one_past_end_idx)
        pooled_tensors_list.append(pooled_tensor)
        j += word_token_length

    final_word_token_length = 1
    if len(word_offsets) - 1 == end_word_offset:
        final_word_token_length = len(tokens_list) - word_offsets[end_word_offset]
    else:
        final_word_token_length = word_offsets[end_word_offset + 1] - word_offsets[end_word_offset]
    final_start_idx = j - start_word_offset
    final_one_past_end_idx = final_start_idx + final_word_token_length
    #print(final_start_idx, final_one_past_end_idx)
    final_pooled_tensor = word_tokens_tensor[final_start_idx:final_one_past_end_idx].mean(dim=0)
    pooled_tensors_list.append(final_pooled_tensor)

    pooled_window_tensors = torch.stack(pooled_tensors_list, dim=0)
    return torch.nn.functional.pad(pooled_window_tensors, (0, 0, 0, 32-pooled_window_tensors.size(0)), value=0)
    #return torch.nn.functional.pad(pooled_window_tensors, (0, 32-pooled_window_tensors.size(0)), value=0)
    